In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp, to_date, lower, trim
from pyspark.sql import functions as F

In [2]:
spark = SparkSession.builder \
    .appName("VenchiDataLoading") \
    .getOrCreate()

## Read Data

In [3]:
# Define Paths
accounts_path = r'c:\Users\rossim\Desktop\venchi_project\venchi_jMLE_candidate_pack\data\accounts.parquet'
interactions_path = r'c:\Users\rossim\Desktop\venchi_project\venchi_jMLE_candidate_pack\data\interactions.parquet'
products_path = r'c:\Users\rossim\Desktop\venchi_project\venchi_jMLE_candidate_pack\data\products.txt'
sales_path = r'c:\Users\rossim\Desktop\venchi_project\venchi_jMLE_candidate_pack\data\sales.parquet'

# 1. Accounts (Parquet preserves schema, but we ensure timestamp type)
print("Accounts")
accounts_df = spark.read.parquet(accounts_path) \
    .withColumn("timestamp", to_timestamp(col("timestamp")))
accounts_df.printSchema()

# 2. Interactions
print("Interactions")
interactions_df = spark.read.parquet(interactions_path) \
    .withColumn("date", to_date(col("date")))
interactions_df.printSchema()

# 3. Sales
print("Sales")
sales_df = spark.read.parquet(sales_path) \
    .withColumn("date", to_date(col("date")))
sales_df.printSchema()

# 4. Products (Reading Tab-Separated TXT)
print("Products")
products_df = spark.read.csv(products_path, sep='\t', header=True, inferSchema=True)
products_df.printSchema()

Accounts
root
 |-- account_id: string (nullable = true)
 |-- hct: long (nullable = true)
 |-- staff: long (nullable = true)
 |-- turnover_m_usd: long (nullable = true)
 |-- brand_loyalty: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)

Interactions
root
 |-- interaction_id: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- duration_mins: long (nullable = true)
 |-- response: string (nullable = true)
 |-- topic: string (nullable = true)
 |-- date: date (nullable = true)
 |-- account_id: string (nullable = true)
 |-- product_id: string (nullable = true)

Sales
root
 |-- sale_id: string (nullable = true)
 |-- date: date (nullable = true)
 |-- account_id: string (nullable = true)
 |-- product_id: string (nullable = true)

Products
root
 |-- product_id: string (nullable = true)
 |-- maturity: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- patent_id: string (nullable = true)



## Task 1

### Data Cleaning

In [4]:
accounts_clean = accounts_df.filter(col("account_id").isNotNull())

interactions_clean = interactions_df.filter(col("account_id").isNotNull()) \
    .withColumn("topic_clean", lower(trim(col("topic")))) \
    .withColumn("channel_clean", lower(trim(col("channel")))) \
    .withColumn("response_clean", lower(trim(col("response"))))

sales_clean = sales_df.filter(col("account_id").isNotNull())
products_clean = products_df.filter(col("product_id").isNotNull())

### KPI 1: Total Revenue Generated

In [6]:
sales_with_price = sales_clean.join(products_clean, on="product_id", how="left")
sales_with_price.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- sale_id: string (nullable = true)
 |-- date: date (nullable = true)
 |-- account_id: string (nullable = true)
 |-- maturity: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- patent_id: string (nullable = true)



In [7]:
# Aggregate revenue per customer
kpi_revenue = sales_with_price.groupBy("account_id").agg(
    F.sum("price").alias("total_revenue")
)

In [8]:
kpi_revenue.show(10)

+--------------------+-------------+
|          account_id|total_revenue|
+--------------------+-------------+
|e1672d506568deedf...|       121980|
|cb04cb651869ca904...|        56680|
|a054955a13c04e9aa...|       106884|
|115d1256179d8ca02...|        89245|
|ccaf873af5b6fdd3b...|        93540|
|2aca7152120b6433f...|        39077|
|caf1c34f21405ceb6...|        68992|
|79f393197ee0b3251...|        45490|
|0ff82ad074e663021...|        63555|
|21fc105190774346e...|        63909|
+--------------------+-------------+
only showing top 10 rows


In [9]:
## Extra interesting KPI 

kpi_revenue_product = sales_with_price.groupBy("account_id", "product_id").agg(
    F.sum("price").alias("total_revenue")
)


kpi_revenue_category = sales_with_price.groupBy("account_id", "category").agg(
    F.sum("price").alias("total_revenue")
)

### KPI 2: Number of sales interactions in the last 6 months

In [10]:
max_date_row = interactions_clean.select(F.max("date").alias("latest_date")).collect()[0]
latest_date = max_date_row["latest_date"]
print(f"Latest interaction date: {latest_date}")

Latest interaction date: 2021-02-05


In [11]:
sales_interactions_6m = interactions_clean.filter(
    (F.col("date") >= F.date_sub(F.lit(latest_date), 180))
)
sales_interactions_6m.show(10)

+--------------------+------------+-------------+--------+-------------------+----------+--------------------+--------------------+-------------------+-------------+--------------+
|      interaction_id|     channel|duration_mins|response|              topic|      date|          account_id|          product_id|        topic_clean|channel_clean|response_clean|
+--------------------+------------+-------------+--------+-------------------+----------+--------------------+--------------------+-------------------+-------------+--------------+
|9e8a83833a5a5dafb...|       email|            6|positive|against competition|2020-10-17|544feaa1526ef94f1...|2a38ce238f4b3a52b...|against competition|        email|      positive|
|17f65a62b55523036...|face to face|           82|   mixed|  available finance|2021-01-24|f63ebf8ff4a382c2e...|0eed3f6963570b036...|  available finance| face to face|         mixed|
|909f3a6eca8fbf6b7...|face to face|           55|positive|               cost|2020-09-22|149a85

In [12]:
# Aggregate count per customer
kpi_recent_sales_interactions = sales_interactions_6m.groupBy("account_id").agg(
    F.count("interaction_id").alias("sales_interactions_last_6m")
)

In [13]:
kpi_recent_sales_interactions.show(10)

+--------------------+--------------------------+
|          account_id|sales_interactions_last_6m|
+--------------------+--------------------------+
|ae9e95dc1fb78abec...|                         1|
|2b399e65042a6c1b5...|                         1|
|fb8a2c3a0a5044235...|                         1|
|4029ac1cf58212d04...|                         1|
|3cb614754aec72774...|                         1|
|2399c7d6fd5d8a213...|                         1|
|30c501d1b5b4b8fce...|                         1|
|309dae2ef6410ce4a...|                         1|
|7ae20648c2c5e521c...|                         1|
|fdaf5d1016c282454...|                         1|
+--------------------+--------------------------+
only showing top 10 rows


### KPI 3 Distinct Category

In [14]:
kpi_distinct_cateogory = sales_with_price.groupBy("account_id").agg(
    F.countDistinct("category").alias("distinct_categories") 
)

In [15]:
kpi_distinct_cateogory.show(10)

+--------------------+-------------------+
|          account_id|distinct_categories|
+--------------------+-------------------+
|97830d3b951ec86b8...|                  4|
|936ce27e14558596f...|                  2|
|59d117fcbc6b370f7...|                  4|
|e5131a914d7c3b9f7...|                  4|
|54a774e3b8b30834d...|                  4|
|149c86cc1d278a2d5...|                  4|
|98f62e8b624d99682...|                  4|
|a054955a13c04e9aa...|                  4|
|4bf028c37e2c27dfe...|                  4|
|edba0822938758da1...|                  4|
+--------------------+-------------------+
only showing top 10 rows


### KPI: Average interaction for client

In [16]:
kpi_avg_duration = interactions_clean.groupBy("account_id").agg(
    F.round(F.avg("duration_mins"), 2).alias("avg_interaction_duration_mins")
)

In [17]:
kpi_avg_duration.show(5)

+--------------------+-----------------------------+
|          account_id|avg_interaction_duration_mins|
+--------------------+-----------------------------+
|285fffa8d29a9f9b2...|                         14.0|
|26b8c340d56a7e3bb...|                        102.5|
|ae9e95dc1fb78abec...|                         70.5|
|98f62e8b624d99682...|                        121.0|
|fdf8a784a1d444a52...|                       132.67|
+--------------------+-----------------------------+
only showing top 5 rows


## TASK 2

### Putting Data Together

In [18]:
analytical_dataset = accounts_clean \
    .join(kpi_revenue, on="account_id", how="left") \
    .join(kpi_recent_sales_interactions, on="account_id", how="left") \
    .join(kpi_distinct_cateogory, on="account_id", how="left") \
    .join(kpi_avg_duration, on="account_id", how="left")

In [19]:
shape = (analytical_dataset.count(), len(analytical_dataset.columns))
print(shape)

(12205, 10)


In [21]:
analytical_dataset = analytical_dataset.fillna({
    "total_revenue": 0,
    "sales_interactions_last_6m": 0,
    "distinct_categories": 0,
    "avg_interaction_duration_mins": 0
})

In [22]:
analytical_dataset.show(10)

+--------------------+---+-----+--------------+-------------+-------------------+-------------+--------------------------+-------------------+-----------------------------+
|          account_id|hct|staff|turnover_m_usd|brand_loyalty|          timestamp|total_revenue|sales_interactions_last_6m|distinct_categories|avg_interaction_duration_mins|
+--------------------+---+-----+--------------+-------------+-------------------+-------------+--------------------------+-------------------+-----------------------------+
|fbcd0ff0529a3dd9b...| 21|    3|            98|            3|2020-06-07 11:40:37|       106749|                         0|                  4|                        88.33|
|fe3754abdd3566199...| 57|   16|            26|            5|2020-06-07 11:40:37|        61777|                         0|                  4|                        165.0|
|1f503043085b329a5...|  6|   19|            98|            4|2020-06-07 11:40:37|        70769|                         1|             

### Developing the model

In [23]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

In [ ]:

# ==========================================
# 1. PREPARE DATA FOR MACHINE LEARNING
# ==========================================
# Ensure no nulls exist in the original account columns before vectorization.
ml_data = analytical_dataset.fillna(0)

# Define the features we want our model to learn from.
# We mix firmographics (turnover) with our engineered behavioral KPIs.
feature_cols = [
    "turnover_m_usd", 
    "brand_loyalty", 
    "total_revenue", 
    "sales_interactions_last_6m", 
    "distinct_categories", 
    "avg_interaction_duration_mins"
]

# VectorAssembler combines our individual feature columns into a single vector column called 'features'
assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")
assembled_data = assembler.transform(ml_data)

# StandardScaler normalizes the features so they have a mean of 0 and std dev of 1.
# This prevents large numerical values (like revenue) from dominating the K-Means distance calculations.
scaler = StandardScaler(inputCol="raw_features", outputCol="scaled_features", withStd=True, withMean=True)
scaler_model = scaler.fit(assembled_data)
scaled_data = scaler_model.transform(assembled_data)


# ==========================================
# 2. DETERMINE THE NUMBER OF CLUSTERS (k)
# ==========================================
# We iterate through potential cluster sizes (2 to 5) and calculate the Silhouette Score.
# A score closer to 1 indicates well-separated, dense clusters.

evaluator = ClusteringEvaluator(predictionCol="prediction", featuresCol="scaled_features", metricName="silhouette")
best_k = 2
best_score = -1

print("Evaluating cluster sizes...")
for k in range(2, 6):
    kmeans = KMeans(featuresCol="scaled_features", k=k, seed=42)
    model = kmeans.fit(scaled_data)
    predictions = model.transform(scaled_data)
    score = evaluator.evaluate(predictions)
    print(f"Silhouette Score for k={k}: {score:.4f}")
    
    if score > best_score:
        best_score = score
        best_k = k

print(f"\nSelected optimal number of clusters: {best_k} (based on highest Silhouette Score)")


# ==========================================
# 3. TRAIN FINAL MODEL & ASSIGN SEGMENTS
# ==========================================
# Train the final model using the best K
final_kmeans = KMeans(featuresCol="scaled_features", k=best_k, seed=42)
final_model = final_kmeans.fit(scaled_data)

# Apply the model to our dataset to generate the 'prediction' column (the segment ID)
enriched_data = final_model.transform(scaled_data)

# Rename 'prediction' to 'customer_segment' for business clarity
enriched_data = enriched_data.withColumnRenamed("prediction", "customer_segment")


# ==========================================
# 4. PROFILE THE SEGMENTS
# ==========================================
# Calculate the average values for our KPIs per segment to understand who these customers are.
segment_profiles = enriched_data.groupBy("customer_segment").agg(
    F.count("account_id").alias("customer_count"),
    F.round(F.avg("turnover_m_usd"), 2).alias("avg_turnover"),
    F.round(F.avg("brand_loyalty"), 2).alias("avg_loyalty"),
    F.round(F.avg("total_revenue"), 2).alias("avg_revenue"),
    F.round(F.avg("sales_interactions_last_6m"), 2).alias("avg_recent_interactions"),
    F.round(F.avg("distinct_categories"), 2).alias("avg_distinct_categories")
).orderBy("customer_segment")

print("\n--- Segment Profiles ---")
segment_profiles.show()


Evaluating cluster sizes...
Silhouette Score for k=2: 0.3671
Silhouette Score for k=3: 0.3008
Silhouette Score for k=4: 0.3458
Silhouette Score for k=5: 0.4230

Selected optimal number of clusters: 5 (based on highest Silhouette Score)

--- Segment Profiles ---
+----------------+--------------+------------+-----------+-----------+-----------------------+-----------------------+
|customer_segment|customer_count|avg_turnover|avg_loyalty|avg_revenue|avg_recent_interactions|avg_distinct_categories|
+----------------+--------------+------------+-----------+-----------+-----------------------+-----------------------+
|               0|          3910|       32.15|       6.87|   47127.48|                    0.0|                   3.99|
|               1|          1394|       17.56|       3.83|   18805.49|                   0.02|                   2.52|
|               2|           803|       60.91|       5.86|   68299.11|                   1.03|                   3.92|
|               3|      

Py4JJavaError: An error occurred while calling o829.parquet.
: java.lang.RuntimeException: java.io.FileNotFoundException: java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset. -see https://cwiki.apache.org/confluence/display/HADOOP2/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:789)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:298)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:314)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:1179)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:861)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:901)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:873)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:900)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdir

In [ ]:
# We drop the ML-specific vector columns before saving to keep the final dataset clean.
final_export_df = enriched_data.drop("raw_features", "scaled_features")

output_path = r"C:\Users\rossim\Desktop\venchi_project\venchi_jMLE_candidate_pack\data\accounts_enriched.parquet"

# Write to Parquet (overwrite if it already exists)
final_export_df.write.mode("overwrite").parquet(output_path)
print(f"\nEnriched dataset saved to: {output_path}")

### Idea of Keeping only the recent Interactions

In [ ]:
analytical_dataset = analytical_dataset.filter(col("sales_interactions_last_6m").isNotNull())

In [ ]:
shape = (analytical_dataset.count(), len(analytical_dataset.columns))
print(shape)

In [ ]:
# Order by total_revenue from highest to lowest
sorted_df = analytical_dataset.orderBy(F.col("sales_interactions_last_6m").desc())

sorted_df.show(5)